# Experiment B.02 — freeze variants, manifest, and reset pairing
Do not continue unless exactly three frozen-task variants are printed and reviewed.

In [ ]:
import csv,hashlib,json,os,subprocess
from pathlib import Path
GPU="4"
R=Path.home()/"async-vla-latency-bench"; P=Path.home()/"LIBERO-plus"; OUT=Path.home()/"experiment_b"; EA=Path.home()/"experiment_a"; GATE=EA/"analysis/experiment_a_to_b_gate.json"; N=Path.home()/"stage1-native"; ID=Path.home()/"venv-stage1-id/bin/python"; OOD=Path.home()/"venv-stage1-ood/bin/python"
VAR=OUT/"experiment_b_frozen_object_layout_variants.csv"
if VAR.exists(): print("Using already-frozen Experiment B variant CSV; refusing reselection")
else:
 env=os.environ.copy(); env.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"PYTHONPATH":str(P),"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","MAGICK_HOME":str(N),"PATH":str(N/"bin")+os.pathsep+env.get("PATH",""),"LD_LIBRARY_PATH":str(N/"lib")+os.pathsep+env.get("LD_LIBRARY_PATH","")})
 subprocess.run([str(OOD),"-m","async_vla_benchmark.scripts.resolve_experiment_b_variants","--experiment-a-gate",str(GATE),"--output",str(VAR)],cwd=R,env=env,check=True)
rows=list(csv.DictReader(open(VAR))); assert len(rows)==3
print(*rows,sep="\n"); print("STOP HERE: review and freeze these exact 3 Experiment B identities before running the next cell")

In [ ]:
MAN=OUT/"experiment_b_manifest.csv"; AUDIT=OUT/"experiment_b_initialization_pairing_audit.csv"
bench=subprocess.run(["git","-C",str(R),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(); plus=subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip()
base=os.environ.copy(); base.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg"})
subprocess.run([str(ID),"-m","async_vla_benchmark.scripts.make_experiment_b_manifest","--variants",str(VAR),"--experiment-a-gate",str(GATE),"--output",str(MAN),"--git-sha",bench,"--lerobot-git-sha","2aba372b4e217cc47db28e0f836859b20d1456c9","--libero-plus-git-sha",plus,"--model-revision","8e174154ef5f6c60a8da12ae99c303d8963138c1"],cwd=R,env=base,check=True)
subprocess.run([str(ID),"-m","async_vla_benchmark.scripts.resolve_stage3_initializations","--config",str(R/"async_vla_benchmark/configs/experiment_b.yaml"),"--manifest",str(MAN),"--scene","id","--expected-rows","64","--expected-cells-per-key","2","--audit-output",str(AUDIT)],cwd=R,env=base,check=True)
ood=base.copy(); ood.update({"PYTHONPATH":str(P),"MAGICK_HOME":str(N),"PATH":str(N/"bin")+os.pathsep+ood.get("PATH",""),"LD_LIBRARY_PATH":str(N/"lib")+os.pathsep+ood.get("LD_LIBRARY_PATH","")})
subprocess.run([str(OOD),"-m","async_vla_benchmark.scripts.resolve_stage3_initializations","--config",str(R/"async_vla_benchmark/configs/experiment_b.yaml"),"--manifest",str(MAN),"--scene","ood","--expected-rows","64","--expected-cells-per-key","2","--audit-output",str(AUDIT)],cwd=R,env=ood,check=True)
subprocess.run([str(ID),"-m","async_vla_benchmark.scripts.validate_experiment_b","--variants",str(VAR),"--manifest",str(MAN),"--experiment-a-gate",str(GATE),"--output-dir",str(OUT),"--allow-incomplete"],cwd=R,env=base,check=True)
pre=json.loads((OUT/"experiment_b_preflight_environment.json").read_text()); pre.update({"frozen_variant_csv_sha256":hashlib.sha256(VAR.read_bytes()).hexdigest(),"manifest_sha256":hashlib.sha256(MAN.read_bytes()).hexdigest(),"spec_sha256":hashlib.sha256((R/"docs/EXPERIMENT_B_ADDITIONAL_MULTI_STAGE_TASK_GENERALIZATION.md").read_bytes()).hexdigest()}); (OUT/"experiment_b_preflight_environment.json").write_text(json.dumps(pre,indent=2)+"\n")
print("PASS: 64-row Experiment B manifest and 32 two-delay reset-pairing identities frozen")